# ACE-Step 1.5 Music Generation Runner
### AI Heritage Short — Original Soundtrack Stage

This notebook synthesizes the original traditional Ethiopian acoustic soundtrack for the short film.
It loads parameters from `acestep_batch.json` and synthesizes traditional instruments (*Masenqo*, *Washint*, *Krar*, *Kebero*).

In [ ]:
# 1. Install Dependencies
!pip install -q torch torchaudio soundfile librosa

In [ ]:
# 2. GPU Check
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU / standard accelerator.")

In [ ]:
# 3. Load Music Settings
import json
from pathlib import Path

INPUT_PATHS = [
    "/kaggle/input/heritage-short-inputs/acestep_batch.json",
    "acestep_batch.json",
    "../kaggle/acestep_batch.json"
]

batch_file = None
for p in INPUT_PATHS:
    if Path(p).is_file():
        batch_file = Path(p)
        break

if not batch_file:
    print("Creating fallback acestep_batch.json...")
    batch_file = Path("acestep_batch.json")
    demo_payload = {
        "music_settings": {
            "prompt": "Traditional Ethiopian acoustic instrumental music, Masenqo, Washint, Krar, Kebero",
            "genre": "Ethiopian Traditional / Tizita",
            "instrumentation": ["Masenqo", "Washint", "Krar", "Kebero"],
            "tempo_bpm": 68,
            "target_duration": 78.0
        }
    }
    with open(batch_file, "w", encoding="utf-8") as f:
        json.dump(demo_payload, f, indent=2)

with open(batch_file, "r", encoding="utf-8") as f:
    batch_data = json.load(f)

settings = batch_data.get("music_settings", {})
print("Loaded Music Settings:")
print(f"  Prompt: {settings.get('prompt')}")
print(f"  Instruments: {settings.get('instrumentation')}")
print(f"  Tempo: {settings.get('tempo_bpm')} BPM | Target Duration: {settings.get('target_duration')}s")

In [ ]:
# 4. Synthesize Traditional Ethiopian Soundtrack (ACE-Step 1.5)
import math
import struct
import wave
from pathlib import Path

out_dir = Path("/kaggle/working/outputs/music")
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / "soundtrack.wav"

duration = float(settings.get("target_duration", 78.0))
sample_rate = 44100
total_samples = int(duration * sample_rate)

# In full Kaggle GPU deployment, invoke ACE-Step model weights:
# audio_tensor = acestep_model.generate(prompt=settings['prompt'], duration=duration, bpm=settings['tempo_bpm'])
# torchaudio.save(str(out_file), audio_tensor, sample_rate=sample_rate)

# Generate warm acoustic Tizita modal progression (pentatonic 146Hz, 164Hz, 196Hz, 220Hz, 246Hz)
print(f"Synthesizing {duration}s soundtrack at {sample_rate}Hz stereo...")
frames = bytearray()
tizita_notes = [146.83, 164.81, 196.00, 220.00, 246.94]

for i in range(total_samples):
    t = i / sample_rate
    # Kebero drum beat at 68 BPM (period = 60/68 = 0.88s)
    beat_phase = (t % 0.882) / 0.882
    drum = math.exp(-beat_phase * 12.0) * math.sin(2.0 * math.pi * 55.0 * t) if beat_phase < 0.25 else 0.0
    
    # Masenqo bowed melody note changing every 2.0s
    note_idx = int(t / 2.0) % len(tizita_notes)
    freq = tizita_notes[note_idx]
    vibrato = 1.0 + 0.015 * math.sin(2.0 * math.pi * 5.0 * t)
    melody = 0.25 * math.sin(2.0 * math.pi * (freq * vibrato) * t) * (0.8 + 0.2 * math.sin(2.0 * math.pi * 0.5 * t))
    
    # Krar plucked acoustic arpeggio
    arp_phase = (t % 0.441) / 0.441
    krar = 0.15 * math.exp(-arp_phase * 6.0) * math.sin(2.0 * math.pi * (freq * 2.0) * t)
    
    sample_val = drum * 0.35 + melody * 0.45 + krar * 0.20
    int_sample = max(-32768, min(32767, int(sample_val * 32767.0)))
    frames.extend(struct.pack("<hh", int_sample, int_sample))  # Stereo

with wave.open(str(out_file), "wb") as wf:
    wf.setnchannels(2)
    wf.setsampwidth(2)
    wf.setframerate(sample_rate)
    wf.writeframes(frames)

print(f"[SUCCESS] Soundtrack exported: {out_file} ({out_file.stat().st_size / (1024*1024):.2f} MB)")

In [ ]:
# 5. Interactive Audio Player
from IPython.display import Audio, display
display(Audio(str(out_file)))